# 02 · Debugging (`pdb`, `breakpoint()`, IDE) & Logging

**Module 07 – Errors, Debugging & Testing** · Estimated time: 45 min

> Extends 📗 *Python Debugger (pdb)* (from Advanced Python Modules). **Logging** is missing from both sources and is essential for real programs.

### 🎯 What you will learn
- A debugging mindset: reproduce → isolate → inspect → fix → test
- Quick inspection with `print` and `f"{var=}"`
- The Python debugger: `pdb`, `breakpoint()`, the key commands
- Post-mortem debugging and IDE debuggers
- The `logging` module: levels, formatting, files, per-module loggers
- `warnings` and `traceback`

## 1. A debugging mindset
1. **Reproduce** the bug reliably
2. **Read** the traceback carefully (bottom-up)
3. **Isolate**: shrink the input/code until the bug is obvious
4. **Inspect** values at the point of failure (print, debugger, logs)
5. **Fix**, then **add a test** so it never comes back

## 2. Print debugging (quick and effective)

In [1]:
def average(values):
    total = sum(values)
    print(f"{values=} {total=} {len(values)=}")    # 3.8+ self-documenting f-strings
    return total / len(values)

average([3, 5, 10])

values=[3, 5, 10] total=18 len(values)=3


6.0

## 3. The Python debugger: `pdb` (📗)

The bootcamp example: we want to find out *why* this fails:

In [2]:
x = [1, 3, 4]
y = 2
z = 3

result = y + z
print(result)
result2 = y + x          # TypeError
print(result2)

5


TypeError: unsupported operand type(s) for +: 'int' and 'list'

Insert a **breakpoint** to pause execution and explore the variables interactively:

In [ ]:
x = [1, 3, 4]
y = 2
z = 3

result = y + z
print(result)

breakpoint()             # ✅ modern (3.7+). ⚠️ Old style: import pdb; pdb.set_trace()

result2 = y + x
print(result2)

> ⌨️ This cell isn't pre-run because it waits for your input. Run it, type the commands below in the prompt, and type `q` to quit.

### Essential pdb commands

| Command | Meaning |
|---|---|
| `p expr` / `pp expr` | print / pretty-print an expression (e.g. `p x`, `p type(y)`) |
| `l` / `ll` | list the code around the current line / the whole function |
| `n` | **next** line (step *over* function calls) |
| `s` | **step into** a function call |
| `c` | **continue** until the next breakpoint |
| `r` | continue until the current function **returns** |
| `b 25` | set a breakpoint at line 25 (`b` alone lists breakpoints) |
| `w` | **where**: show the call stack |
| `u` / `d` | move up/down the call stack |
| `!stmt` | execute a Python statement (e.g. `!x = [0]`) |
| `interact` | open a full Python shell with the current variables |
| `q` | quit |

`breakpoint()` respects the `PYTHONBREAKPOINT` environment variable; `PYTHONBREAKPOINT=0` disables all breakpoints without editing code.

### Other ways to start pdb
```bash
python -m pdb my_script.py          # start a script under the debugger
python -m pdb -p 12345              # 🆕 3.14: attach to a running Python process by PID
```
In Jupyter, after an exception, run **`%debug`** in a new cell to inspect the crash (*post-mortem* debugging).

## 4. IDE debuggers (recommended for everyday work)
In **VS Code** / **PyCharm**: click left of a line number to set a red-dot breakpoint, then *Run → Start Debugging* (F5). You get variable panes, watch expressions, call stack and step buttons, i.e. pdb with a GUI. VS Code can debug Jupyter cells too.

## 5. Logging: `print()` for grown-ups

`print` statements get deleted before shipping; logs **stay** and can be turned up or down without changing code.

### Log levels

| Level | Number | Use for |
|---|---|---|
| `DEBUG` | 10 | detailed diagnostic info |
| `INFO` | 20 | normal events ("user logged in") |
| `WARNING` | 30 | something unexpected, but the program continues (the **default** threshold) |
| `ERROR` | 40 | an operation failed |
| `CRITICAL` | 50 | the program may not be able to continue |

In [3]:
import logging, sys

# Configure once, at program start-up. force=True resets the config so this cell can be re-run.
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
    force=True,
)

logging.debug("Debug details: x=%s", 42)
logging.info("Application started")
logging.warning("Disk space is low")
logging.error("Could not save the file")
logging.critical("Database unreachable!")

19:42:26 | DEBUG    | root | Debug details: x=42


19:42:26 | INFO     | root | Application started


19:42:26 | WARNING  | root | Disk space is low


19:42:26 | ERROR    | root | Could not save the file


19:42:26 | CRITICAL | root | Database unreachable!


💡 Note the `%s` style: logging formats lazily, so the string is only built if the message is actually emitted. (This is the one place where old `%`-formatting is still recommended.)

### Per-module loggers (best practice)

In [4]:
logger = logging.getLogger("shop.orders")      # in real modules: logging.getLogger(__name__)

def place_order(item, qty):
    logger.info("Placing order: %s x%d", item, qty)
    if qty <= 0:
        logger.warning("Rejected order with qty=%d", qty)
        return False
    return True

place_order("pen", 3)
place_order("pen", 0)

19:42:26 | INFO     | shop.orders | Placing order: pen x3


19:42:26 | INFO     | shop.orders | Placing order: pen x0


19:42:26 | WARNING  | shop.orders | Rejected order with qty=0


False

### Logging exceptions with the traceback

In [5]:
try:
    1 / 0
except ZeroDivisionError:
    logger.exception("Calculation failed")      # logs at ERROR level + full traceback

19:42:26 | ERROR    | shop.orders | Calculation failed
Traceback (most recent call last):
  File "/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/ipykernel_5480/2501632117.py", line 2, in <module>
    1 / 0
    ~~^~~
ZeroDivisionError: division by zero


### Logging to a file

In [6]:
file_logger = logging.getLogger("audit")
handler = logging.FileHandler("app.log", encoding="utf-8")
handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
file_logger.addHandler(handler)
file_logger.propagate = False              # don't also print to the console
file_logger.warning("User asha changed password")
handler.close()
print(open("app.log", encoding="utf-8").read())

2026-10-20 19:42:26,177 WARNING User asha changed password



### Changing verbosity without touching code

In [7]:
logging.getLogger().setLevel(logging.WARNING)   # silence DEBUG/INFO everywhere
logger.info("you won't see this")
logger.warning("but you will see this")

19:42:26 | WARNING  | shop.orders | but you will see this


## 6. `warnings`: soft alerts (e.g. deprecations)

In [8]:
import warnings

def old_function():
    warnings.warn("old_function() is deprecated; use new_function()", DeprecationWarning, stacklevel=2)
    return "result"

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    old_function()
print(caught[0].category.__name__, "-", caught[0].message)

DeprecationWarning - old_function() is deprecated; use new_function()


> 🆕 Python 3.13 added the **`@warnings.deprecated`** decorator, which also lets type checkers flag usages of deprecated functions:

In [9]:
from warnings import deprecated

@deprecated("Use add_v2() instead")
def add_v1(a, b):
    return a + b

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    add_v1(1, 2)
print(caught[0].message)

Use add_v2() instead


## 7. The `traceback` module

In [10]:
import traceback
try:
    {}["missing"]
except KeyError:
    text = traceback.format_exc()
print(text)

Traceback (most recent call last):
  File "/var/folders/96/f6xk04z91_gdystl3qfqv33r0000gn/T/ipykernel_5480/1707355043.py", line 3, in <module>
    {}["missing"]
    ~~^^^^^^^^^^^
KeyError: 'missing'



## 📝 Summary
- Debug systematically; `f"{x=}"` for quick looks.
- `breakpoint()` (or IDE breakpoints) to pause and inspect; learn `n s c p l w q`.
- `%debug` in Jupyter for post-mortem; `python -m pdb -p PID` (3.14) attaches to running processes.
- Use `logging` with levels and per-module loggers instead of `print` in real programs.
- `logger.exception()` inside `except` logs the traceback.

## 🏋️ Practice
1. Run the `breakpoint()` cell above; use `p x`, `p type(y)`, `n`, and `q`.
2. Configure logging to write `INFO` and above to `myapp.log` with timestamps.
3. Write `divide(a, b)` that logs a warning when `b == 0` and returns `None`.

<details><summary>Show solutions</summary>

```python
import logging
logging.basicConfig(filename="myapp.log", level=logging.INFO,
                    format="%(asctime)s %(levelname)s %(message)s", force=True)

log = logging.getLogger(__name__)
def divide(a, b):
    if b == 0:
        log.warning("Division by zero attempted: a=%s", a)
        return None
    return a / b
```
</details>